In [0]:
-- SOURCES RAW COUNTS :param_1

Select 'bronze_customers' As table_name, count(*) as row_count from workspace.default.bronze_customers
union
select 'bronze_orders', count(*) from  workspace.default.bronze_orders
union
select 'bronze_employees', count(*) from workspace.default.bronze_employees
union
select 'bronze_products', count(*) from workspace.default.bronze_products
union
select 'bronze_stores'  , count(*) from workspace.default.bronze_stores
union
select 'bronze_returns' , count(*) from workspace.default.bronze_returns
union
select 'bronze_inventory_snapshots' , count(*) from workspace.default.bronze_inventory_snapshots
union 
select 'bronze_customer_segment_bridge' , count(*)  from workspace.default.bronze_customer_segment_bridge
union
select 'bronze_order_items', count(*) from workspace.default.bronze_order_items;


------- TABLE AUDITS

-- CUSTOMERS 

Select * from workspace.default.bronze_customers ;

SELECT DISTINCT source_system
FROM workspace.default.bronze_customers;

SELECT source_system, COUNT(*) AS cnt
FROM workspace.default.bronze_customers
GROUP BY source_system
ORDER BY cnt DESC;

-- ORDERS

SELECT * FROM workspace.default.bronze_orders limit 10;

SELECT DISTINCT order_status FROM workspace.default.bronze_orders;


SELECT order_status, COUNT(order_id) AS customers_count
FROM workspace.default.bronze_orders
GROUP BY order_status
ORDER BY customers_count;

SELECT snapshot_date, COUNT(*) AS store_product_combinations
FROM workspace.default.bronze_inventory_snapshots
GROUP BY snapshot_date
ORDER BY snapshot_date;

-- Confirm source_file is populated and points to correct folder
SELECT DISTINCT source_file, ingested_at
FROM workspace.default.bronze_customers;

SELECT * FROM workspace.default.bronze_order_items;

select distinct source_file, ingested_at from workspace.default.bronze_order_items;


-- Comparison of counts of Data in Bronze and Silver
-- Lets see how our data quality and quality has changed on constraints 
-- and validation checks

SELECT
(SELECT 'Customers' as TableName),
(SELECT count(*) from workspace.default.bronze_employees) as bronze,
(SELECT count(*) from default.silver_employees) as silver,
(bronze-silver) as DeltaCount
Union ALL
SELECT
(SELECT 'Products' as TableName),
(SELECT count(*) from workspace.default.bronze_products) as bronze,
(SELECT count(*) from workspace.default.silver_products) as silver,
(bronze-silver) as DeltaCount
Union ALL
SELECT
(SELECT 'Orders' as TableName),
(SELECT count(*) from workspace.default.bronze_orders) as bronze,
(SELECT count(*) from workspace.default.silver_orders) as silver,
(bronze-silver) as DeltaCount
Union ALL
SELECT
(SELECT 'Order Items' as TableName),
(SELECT count(*) from workspace.default.bronze_order_items) as bronze,
(SELECT count(*) from workspace.default.silver_order_items) as silver,
(bronze-silver) as DeltaCount
Union ALL
SELECT
(Select 'Stores'),
(SELECT count(*) from workspace.default.bronze_stores) as bronze,
(SELECT count(*) from workspace.default.silver_stores) as silver,
(bronze-silver) as DeltaCount
UNION ALL
SELECT 
(SELECT 'Returns'),
(SELECT count(*) from workspace.default.bronze_returns) as bronze,
(SELECT count(*) from workspace.default.silver_returns) as silver,
(bronze-silver) as DeltaCount
Union ALL
SELECT
(SELECT 'Inventory Snapshots' as TableName),
(SELECT count(*) from workspace.default.bronze_inventory_snapshots) as bronze,
(SELECT count(*) from workspace.default.silver_inventory_snapshots) as silver,
(bronze-silver) 
Union ALL
SELECT
(SELECT 'Customer Segments' as TableName),
(SELECT count(*) from workspace.default.bronze_customer_segment_bridge) as bronze,
(SELECT count(*) from workspace.default.silver_customer_segment_bridge) as silver,
(bronze-silver) as Countsy;


--- Null Checks on Silver Order

SELECT 
SUM (CASE WHEN order_id is null then 1 ELSE 0 END) AS order_id_null_count,
SUM (CASE WHEN customer_src_id is null then 1 ELSE 0 END) AS customer_src_id_null_count,
SUM (CASE WHEN store_id is null then 1 ELSE 0 END) AS store_id_null_count,
SUM (CASE WHEN employee_id is null then 1 ELSE 0 END) AS employee_id_null_count,
SUM (CASE WHEN order_date is null then 1 ELSE 0 END) AS order_date_null_count,
SUM (CASE WHEN ship_date is null then 1 ELSE 0 END) AS ship_date_null_count,
SUM (CASE WHEN delivery_date is null then 1 ELSE 0 END) AS delivery_date_null_count,
SUM (CASE WHEN order_status is null then 1 ELSE 0 END) AS order_status_null_count,
SUM (CASE WHEN payment_method is null then 1 ELSE 0 END) AS payment_method_null_count
FROM default.silver_orders;


-- Works fine on round 2
SELECT  order_item_id, order_id, product_src_id, quantity, unit_price_at_sale, discount_pct, net_amount, 
round((quantity*unit_price_at_sale)*(1-discount_pct),3) as expected_net_amt,
CASE WHEN expected_net_amt = net_amount THEN 'OK' ELSE 'ERROR' END as validation_status from default.silver_order_items;


select distinct order_status from silver_orders;
select * from bronze_products;

SELECT 
SUM(CASE WHEN unit_price <0 THEN 1 ELSE 0 END)  as bad_prices,
SUM (CASE WHEN unit_cost < 0 THEN 1 ELSE 0 END) as bad_costs,
SUM (CASE WHEN unit_price<=unit_cost THEN 1 ELSE 0 END) as margin_check
from default.silver_products;

SELECT
  SUM(CASE WHEN unit_price <= 0 THEN 1 ELSE 0 END) AS bad_prices,
  SUM(CASE WHEN unit_cost  <= 0 THEN 1 ELSE 0 END) AS bad_costs,
  SUM(CASE WHEN unit_price <= unit_cost THEN 1 ELSE 0 END) AS negative_margin
FROM workspace.default.silver_products;


SELECT o.* , s.*
FROM workspace.default.silver_orders o
LEFT JOIN workspace.default.silver_stores s ON o.store_id = s.store_id
WHERE s.store_id IS NULL;

select * from silver_returns;
SELECT * FROM silver_order_items;

SELECT r.* from silver_returns r
where r.order_item_id not in (select order_item_id from silver_order_items);

SELECT COUNT(*) AS orphan_return_refs
FROM workspace.default.silver_returns r
LEFT JOIN workspace.default.silver_order_items oi ON r.order_item_id = oi.order_item_id
WHERE oi.order_item_id IS NULL;


-- Unique Order & Order Items ID

-- order_id must be unique
SELECT order_id, COUNT(*) AS cnt
FROM workspace.default.silver_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

-- order_item_id must be unique
SELECT order_item_id, COUNT(*) AS cnt
FROM workspace.default.silver_order_items
GROUP BY order_item_id
HAVING COUNT(*) > 1;

-- Order classification based on order amount. 
-- Nested CASE

SELECT so.order_id, so.order_status,
sum(quantity) as CountOfItems, sum(net_amount) as TotalRevenue, Avg(net_amount) as AvgValue, Min(net_amount) as MinValue, max(net_amount) as MaxValue,
CASE WHEN sum(soi.net_amount)<50 Then 'Low' Else CASE WHEN sum(soi.net_amount)<200 and sum(soi.net_amount)>=50 THEN 'Medium' Else CASE WHEN sum(soi.net_amount)>=200  AND sum(soi.net_amount)<=500 THEN 'Large' ELSE 'Enterprise' END END END as OrderClassification
from silver_orders so 
LEFT JOIN silver_order_items soi ON so.order_id = soi.order_id
group by so.order_id, so.order_status; 

select so.*, soi.* from silver_orders so 
left join  silver_order_items soi 
on so.order_id=soi.order_id
where so.order_id=1822;


-- Employee ranking based on hire date for each store

SELECT employee_id, employee_name, store_id, hire_date, date_diff(YEAR, hire_date, current_date()) as NoOfYears,  
rank() over( order by Hire_Date asc) as EmployeeRank FROM silver_employees; 